# TIR-JDE — model karşılaştırma pipeline'ı (Colab)

Strong Baseline ve Dist-Tracker'ı **aynı veri, aynı split ve aynı değerlendirme** ile eğitir, test eder ve
her şeyi MLflow'a (DagsHub) kaydeder.

**Önce:** *Runtime → Change runtime type → A100* (Strong Baseline'ın flash-attn'i Ampere ister).
**Gizli anahtar** (soldaki 🔑 Secrets): `DAGSHUB_TOKEN` (DagsHub → Settings → Tokens). Tanımlı değilse kayıtlar Drive'daki
`mlflow.db` dosyasına yerel olarak yazılır.

Akış: kurulum → veri → doğrulama grubu → preprocess → tam eğitim (3 seed) → karşılaştırma.

**Run all kullanmayın:** 9. hücre saatler süren tam eğitimi başlatır. Hücreleri sırayla çalıştırın.

In [ ]:
# 1) Drive ve klasörler
from google.colab import drive
drive.mount('/content/drive')

import os
WORK = '/content/drive/MyDrive/tir-jde-work'          # ağırlıklar, çıktılar, yerel MLflow kaydı (kalıcı)
BSB_ZIP = '/content/drive/MyDrive/datasets/BSB/Train.zip'  # Zenodo'dan bir kez indirilir (kalıcı)
BSB_ROOT = '/content/data/BSB'                         # zip her oturumda yerel diske açılır (hızlı)
os.makedirs(WORK, exist_ok=True)
os.environ.update(TIRJDE_WORK=WORK, BSB_ROOT=BSB_ROOT, TIRJDE_BASELINES='/content/baselines',
                  TIRJDE_DATA='/content/work-data')     # preprocess çıktıları da yerel diskte

try:  # DagsHub (MLflow) erişimi Colab Secrets'tan; yalnızca token gerekir
    from google.colab import userdata
    os.environ['DAGSHUB_TOKEN'] = userdata.get('DAGSHUB_TOKEN')
    print('MLflow -> DagsHub')
except Exception:
    print('DagsHub anahtarı yok: MLflow kayıtları', WORK + '/mlflow.db', 'altına yazılacak')
!nvidia-smi --query-gpu=name,memory.total --format=csv
gpu = !nvidia-smi --query-gpu=name --format=csv,noheader
if not any('A100' in g for g in gpu):
    print('\n!!! UYARI: GPU A100 değil (' + ', '.join(gpu) + ').')
    print('!!! Baseline ortamları (torch 2.2.2 / 2.4.1) yeni GPU mimarilerini (ör. Blackwell) desteklemez;')
    print('!!! eğitim "no kernel image" hatasıyla durur. Runtime -> Change runtime type -> A100 seçin.')

In [ ]:
# 2) TIR-JDE ve pipeline bağımlılıkları (Colab'ın kendi Python'u)
%cd /content
!test -d TIR-JDE && git -C TIR-JDE pull -q || git clone -q https://github.com/silaylc/TIR-JDE.git
%cd /content/TIR-JDE
!pip install -q mlflow pyyaml loguru python-dotenv tqdm typer opencv-python-headless matplotlib scipy pandas \
    "git+https://github.com/JonathonLuiten/TrackEval.git" && pip install -q -e . --no-deps
!git log -1 --format='TIR-JDE %h %s'

In [ ]:
# 3) Baseline depoları (configs/model/*.yaml'daki commit'lere sabitlenir)
import yaml, subprocess
for name in ('strong_baseline', 'dist_tracker'):
    m = yaml.safe_load(open(f'configs/model/{name}.yaml'))['model']
    dest = f"/content/baselines/{name}"
    if not os.path.isdir(dest):
        subprocess.run(['git', 'clone', '-q', m['repo'], dest], check=True)
    subprocess.run(['git', '-C', dest, 'checkout', '-q', m['commit']], check=True)
    print(name, subprocess.run(['git', '-C', dest, 'log', '-1', '--format=%h %s'], capture_output=True, text=True).stdout)

In [ ]:
# 4) Strong Baseline ortamı: Python 3.11 + torch 2.2.2 (CUDA 11.8) + flash-attn 2.7.3 (yazarın README'si)
!pip install -q uv
!uv python install 3.11 3.8
!uv venv /content/envs/sb --python 3.11 --seed --clear -q
SB = '/content/envs/sb/bin/python'
!uv pip install -q --python {SB} torch==2.2.2 torchvision==0.17.2 --index-url https://download.pytorch.org/whl/cu118
%cd /content/baselines/strong_baseline/BoT-SORT/yolov12
!wget -q -nc https://github.com/Dao-AILab/flash-attention/releases/download/v2.7.3/flash_attn-2.7.3+cu11torch2.2cxx11abiFALSE-cp311-cp311-linux_x86_64.whl
!uv pip install -q --python {SB} -r requirements.txt
%cd ..
!uv pip install -q --python {SB} -r requirements.txt ultralytics==8.3.88 cython faiss-cpu seaborn
!uv pip install -q --python {SB} --no-build-isolation cython_bbox "git+https://github.com/cocodataset/cocoapi.git#subdirectory=PythonAPI"
!{SB} -c "import torch, flash_attn; print('SB:', torch.__version__, torch.version.cuda, 'flash-attn', flash_attn.__version__); print('GPU testi:', (torch.ones(2, device='cuda') * 2).sum().item(), torch.cuda.get_device_name())"
os.environ['SB_PYTHON'] = SB
%cd /content/TIR-JDE

In [ ]:
# 5) Dist-Tracker ortamı: Python 3.8 + torch 2.4.1 (yazarın environment.yml'i, pip'teki 'ultralytics' HARİÇ)
!uv venv /content/envs/dist --python 3.8 --seed --clear -q
DIST = '/content/envs/dist/bin/python'
!uv pip install -q --python {DIST} torch==2.4.1 torchvision==0.19.1 numpy==1.24.4 opencv-python==4.11.0.86 \
    pillow==10.4.0 pyyaml==6.0.2 requests==2.32.3 scipy==1.10.1 matplotlib==3.7.5 pandas==2.0.3 seaborn==0.13.2 \
    tqdm==4.67.1 psutil==7.0.0 py-cpuinfo==9.0.0 ultralytics-thop==2.0.14 lap==0.5.12
# Yazarın fork'u yerine pip paketi yüklenirse NWD/SSQ kaybı sessizce devre dışı kalır:
!{DIST} -m pip uninstall -y -q ultralytics 2>/dev/null; echo "pip ultralytics yok: OK"
!{DIST} -c "import torch; print('Dist:', torch.__version__, torch.cuda.is_available()); print('GPU testi:', (torch.ones(2, device='cuda') * 2).sum().item(), torch.cuda.get_device_name())"
os.environ['DIST_PYTHON'] = DIST

## Veri: BSB eğitim seti
[Zenodo 15853476](https://zenodo.org/records/15853476) — `Train.zip` (3.8 GB, CC BY 4.0): 102 sekans, 77.293 kare.
İçinde `TrainImages/<seq>/00000.jpg…`, `TrainLabels/<seq>.txt` (MOT formatı) ve `TrainVideos/<seq>.mp4` var.
Zip Drive'a **bir kez** indirilir; her oturumda yerel diske açılır (Drive'da on binlerce küçük dosya çok yavaştır).

In [ ]:
# 6) BSB'yi indir (yalnızca ilk sefer), md5 kontrolü, yerel diske aç
import hashlib, pathlib
URL = 'https://zenodo.org/records/15853476/files/Train.zip?download=1'
MD5 = 'd888235326ea6c939bbdb65258b8b407'
os.makedirs(os.path.dirname(BSB_ZIP), exist_ok=True)
if not os.path.exists(BSB_ZIP):
    !wget -q --show-progress -O "{BSB_ZIP}.part" "{URL}" && mv "{BSB_ZIP}.part" "{BSB_ZIP}"
md5 = hashlib.md5()
with open(BSB_ZIP, 'rb') as f:
    for chunk in iter(lambda: f.read(1 << 24), b''):
        md5.update(chunk)
assert md5.hexdigest() == MD5, 'Train.zip bozuk: dosyayı silip hücreyi tekrar çalıştırın'
if not os.path.isdir(f'{BSB_ROOT}/TrainLabels'):
    !mkdir -p /content/data/tmp && unzip -q -o "{BSB_ZIP}" -d /content/data/tmp
    top = [d for d in pathlib.Path('/content/data/tmp').rglob('TrainLabels') if d.is_dir()][0].parent
    !rm -rf "{BSB_ROOT}" && mv "{top}" "{BSB_ROOT}"
!ls $BSB_ROOT

# Sekans listesi ve kare/etiket tutarlılığı
labels = sorted(p.stem for p in pathlib.Path(BSB_ROOT, 'TrainLabels').glob('*.txt'))
print(len(labels), 'sekans:', labels[:5], '...')
bad = []
for seq in labels:
    n_img = len(list(pathlib.Path(BSB_ROOT, 'TrainImages', seq).glob('*.jpg')))
    last = max(int(l.split(',')[0]) for l in open(f'{BSB_ROOT}/TrainLabels/{seq}.txt') if l.strip())
    if last > n_img:
        bad.append((seq, n_img, last))
print('kare sayısı etiketle uyumsuz:', bad or 'yok')

split = pathlib.Path('data/splits/bsb_train.txt')
if not split.exists():
    split.write_text('\n'.join(labels) + '\n')
    print('data/splits/bsb_train.txt oluşturuldu: bu listeyi repoya ekleyin')

In [ ]:
# 6c) İSTEĞE BAĞLI (bir kez yapıldı, sonuçlar repoda: data/splits/bsb_track3_map.csv)
#     BSB eğitim <-> Track 3 eşleştirmesi, etiket farkları, doğrulamada sahne ikizi, sahne kümeleri.
#     Track 3 eğitim verisi ve EDA çıktıları Drive'da olmalı (yollar aşağıda).
import numpy as np, pandas as pd, pathlib, zipfile
from scipy.optimize import linear_sum_assignment
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components

T3   = pathlib.Path('/content/drive/MyDrive/TIR-JDE/data/track3/MultiUAV_Train/TrainLabels')
EDA  = pathlib.Path('/content/drive/MyDrive/TIR-JDE/outputs/eda')
T3SP = pathlib.Path('/content/drive/MyDrive/TIR-JDE/data/splits')
BSBL = pathlib.Path(BSB_ROOT) / 'TrainLabels'
if not BSBL.is_dir():  # 6. hücre bu oturumda çalışmadıysa yalnızca etiketleri aç
    with zipfile.ZipFile(BSB_ZIP) as zf:
        uyeler = [n for n in zf.namelist() if '/TrainLabels/' in '/' + n and n.endswith('.txt')]
        zf.extractall('/content/data/lbl', members=uyeler)
    BSBL = next(pathlib.Path('/content/data/lbl').rglob('TrainLabels'))

def oku(p):
    d = pd.read_csv(p, header=None, sep=r'[,\s]+', engine='python')
    return d.iloc[:, :6].to_numpy(float) if len(d) else np.zeros((0, 6))
def liste(p):
    return [l.split()[0] for l in open(p) if l.strip() and not l.startswith('#')]
def kutular(a, f):
    return a[a[:, 0] == f][:, 2:6]
def esles(A, B, tol=1.0):  # merkezleri tol pikselden yakın eşleşen kutu sayısı
    if len(A) == 0 or len(B) == 0: return 0
    ca, cb = A[:, :2] + A[:, 2:] / 2, B[:, :2] + B[:, 2:] / 2
    D = np.hypot(ca[:, None, 0] - cb[None, :, 0], ca[:, None, 1] - cb[None, :, 1])
    r, c = linear_sum_assignment(D)
    return int((D[r, c] <= tol).sum())
def skor(a, b):  # 8 örnek karede eşleşen kutu oranı
    fr = np.unique(a[:, 0]); orn = fr[np.linspace(0, len(fr) - 1, 8).astype(int)]
    return sum(esles(kutular(a, f), kutular(b, f)) for f in orn) / max(sum(len(kutular(a, f)) for f in orn), 1)

bsb = {p.stem: oku(p) for p in sorted(BSBL.glob('*.txt'))}
t3 = {s: oku(T3 / f'{s}.txt') for s in liste(T3SP / 'train.txt')}

# 1) eşleştirme
rows = []
for b, a in bsb.items():
    s = sorted(((skor(a, t3[t]), t) for t in t3), reverse=True)
    rows.append(dict(bsb=b, track3=s[0][1], skor=round(s[0][0], 3), ikinci=round(s[1][0], 3)))
m = pd.DataFrame(rows)
iyi = (m.skor >= .9) & ~m.track3.duplicated(keep=False) & (m.ikinci < .5)
print('güvenilir eşleşme:', int(iyi.sum()), '/', len(m))

# 2) BSB ile Track 3 etiket farkları
def fark(a, b):
    return len(a), len(b), sum(esles(kutular(a, f), kutular(b, f), 0.5) for f in np.union1d(a[:, 0], b[:, 0]))
m[['bsb_kutu', 't3_kutu', 'ortak']] = [fark(bsb[r.bsb], t3[r.track3]) for r in m.itertuples()]
m['farkli'] = (m.bsb_kutu != m.ortak) | (m.t3_kutu != m.ortak)
print('etiketi farklı sekans:', m[m.farkli].bsb.tolist() or 'yok')

# 3) sahne kümeleri (parmak izi benzerliği >= 0.999) ve doğrulamadaki en yakın eğitim sahnesi
z = np.load(EDA / 'fingerprints.npz', allow_pickle=True)
ad, V = [str(x) for x in z['names']], z['vecs'].astype(float)
V = V - V.mean(1, keepdims=True); V /= np.maximum(np.linalg.norm(V, axis=1, keepdims=True), 1e-9)
S = pd.DataFrame(V @ V.T, index=ad, columns=ad)
b2t = dict(zip(m.bsb, m.track3)); t2b = {v: k for k, v in b2t.items()}
idx = [b2t[n] for n in bsb]
A = S.loc[idx, idx].to_numpy() >= 0.999; np.fill_diagonal(A, False)
m['scene_group'] = connected_components(csr_matrix(A), directed=False)[1]
val = liste('/content/TIR-JDE/data/splits/val.txt'); egit = [b for b in bsb if b not in val]
print('\nDoğrulama -> eğitimdeki en benzer sahne:')
for v in val:
    s = S.loc[b2t[v], [b2t[e] for e in egit]]
    print(f'  {v:11s} {t2b[s.idxmax()]:11s} {s.max():.4f}')

m.to_csv(f'{WORK}/bsb_track3_train_map.csv', index=False)
print('\nkaydedildi:', f'{WORK}/bsb_track3_train_map.csv')

In [ ]:
# 7) Doğrulama grubu (BİR KEZ). Her ailenin %15'i (en az 1), sahne kümeleri bölünmeden, hatalı sekanslar hariç, sabit seed. Oluşan data/splits/val.txt dosyasını
#    TIR-JDE reposuna commit edin; sonraki tüm deneyler aynı grubu kullanır.
!test -f data/splits/val.txt && cat data/splits/val.txt || python -m tir_jde.pipeline make-val strong_baseline_bsb --fraction 0.15 --seed 0

In [ ]:
# 8) Preprocess: YOLO etiketleri, ReID kırpıntıları, takip girdisi, GT (split hash'ine göre önbelleğe alınır)
!python -m tir_jde.pipeline preprocess strong_baseline_bsb

In [ ]:
# 9) Tam eğitim: 3 seed × 2 model (uzun sürer; Colab kapanırsa aynı hücreyi tekrar çalıştırın,
#     bitmiş seed'ler results.json'dan tanınır ve atlanır)
import pathlib
for exp in ('dist_tracker_bsb', 'strong_baseline_bsb'):
    for seed in (0, 1, 2):
        if pathlib.Path(f'{WORK}/runs/{exp}-s{seed}/results.json').exists():
            print('tamam:', exp, seed); continue
        !python -m tir_jde.pipeline run {exp} seed={seed}

In [ ]:
# 10) Karşılaştırma tablosu ve grafik (ortalama ± std, 3 seed)
!python -m tir_jde.pipeline compare strong_baseline_bsb dist_tracker_bsb --seeds 0 1 2 --out {WORK}/comparison
from IPython.display import Image, display
display(Image(f'{WORK}/comparison/comparison.png'))